# Qwen3.8-Flash-Next NVFP4 on 2 × DGX Spark

## TL;DR

**Measured:** 47.54 output tok/s at concurrency 1, 275.37 aggregate output tok/s at concurrency 16, and 2,960.12 input tok/s for an uncached 16K prompt. This notebook preserves the public measured snapshot and is the runnable controller path for the exact two-node SGLang recipe.

![Measured performance](assets/performance.png)

Evidence class: **MEASURED**. Exact public recipe: [qwen3-8-flash-next-sglang-2x-dgx-spark](https://github.com/PixelML/qwen3-8-flash-next-sglang-2x-dgx-spark/tree/682504bec9e7e99206212f4e172b7ec823e4605c). Limitation: the committed recorded run retains inspected functional verdicts and usage-backed performance, but not the final prompt's generated text; live mode prints a fresh response and complete usage object.

## Requirements

- Two DGX Spark systems on the same system-software release, one GB10 per node, with a working direct RoCE link.
- A Jupyter controller with passwordless SSH aliases to both nodes and Docker access on each node.
- At least 160 GiB of writable node-local model storage on **each** node. Shared or network filesystems fail the preflight.
- Inject `PIXELML_DGX_CONFIG_B64` and `PIXELML_API_TOKEN` through the controller environment; the notebook never publishes or prints their values or storage locations.
- Safety stop: do not continue with a conflicting GPU workload, missing accelerator, storage error, recent Xid, mismatched platform/runtime, inactive interconnect, or core temperature at or above 80 °C.

## Configure

Provide `PIXELML_DGX_NODE_1`, `PIXELML_DGX_NODE_2`, `PIXELML_DGX_CONFIG_B64`, `PIXELML_API_TOKEN`, and `PIXELML_API_BASE` through the environment. The configuration payload follows the pinned detailed recipe's example. Set `PIXELML_RUN_LIVE=1` only when both nodes are authorized for this workload.

In [1]:
from pathlib import Path
import base64, csv, hashlib, importlib.metadata, json, os, re, shlex, subprocess, sys, tempfile, time, urllib.parse, urllib.request

RUN_LIVE = os.environ.get("PIXELML_RUN_LIVE", "0") == "1"
NODE_1 = os.environ.get("PIXELML_DGX_NODE_1", "")
NODE_2 = os.environ.get("PIXELML_DGX_NODE_2", "")
CONFIG_B64 = os.environ.get("PIXELML_DGX_CONFIG_B64", "")
API_TOKEN = os.environ.get("PIXELML_API_TOKEN", "")
API_BASE = os.environ.get("PIXELML_API_BASE", "").rstrip("/")
PROMPT = "Explain why speculative decoding helps single-stream latency."

RECIPE_DIR = Path.cwd().resolve()
if not (RECIPE_DIR / "recipe.json").is_file():
    RECIPE_DIR = (RECIPE_DIR / "recipes" / "qwen3.8-flash-next-sglang").resolve()
REPO_ROOT = RECIPE_DIR.parents[1]
PINS = json.loads((RECIPE_DIR / "recipe.json").read_text(encoding="utf-8"))
RECIPE_SHA = "682504bec9e7e99206212f4e172b7ec823e4605c"
RECIPE_URL = "https://github.com/PixelML/qwen3-8-flash-next-sglang-2x-dgx-spark.git"
LOCKED_PILLOW = "Pillow==11.2.1"
assert LOCKED_PILLOW in (REPO_ROOT / "requirements.lock").read_text(encoding="utf-8").splitlines()
assert importlib.metadata.version("Pillow") == LOCKED_PILLOW.split("==", 1)[1], "install the exact requirements.lock before continuing"
WORK_ROOT = Path(tempfile.mkdtemp())
CONTROLLER_REPO = WORK_ROOT / "pinned-recipe"
NODE_WORKSPACES = {}
CONFIG = {}
print(json.dumps({"mode": "live" if RUN_LIVE else "recorded", "executed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "topology": "2 x DGX Spark, TP=2", "model_revision": PINS["model_revision"], "runtime_pin": PINS["runtime_pin"], "recipe_revision": RECIPE_SHA, "pillow": importlib.metadata.version("Pillow")}, indent=2))

{
  "mode": "recorded",
  "executed_at_utc": "2026-09-01T18:03:51Z",
  "topology": "2 x DGX Spark, TP=2",
  "model_revision": "b80180e371f13348ec49641a6e66999e7854b179",
  "runtime_pin": "PixelML/qwen3-8-flash-next-sglang-2x-dgx-spark@682504bec9e7e99206212f4e172b7ec823e4605c; lmsysorg/sglang@sha256:12d3392bdc8be8d35e9a95f191df6aef99c5114bdbefd41bfdc7e760e6d25ec1",
  "recipe_revision": "682504bec9e7e99206212f4e172b7ec823e4605c",
  "pillow": "11.2.1"
}


## Preflight

Live mode resolves both nodes at runtime and emits only generic pass/fail labels. Each node must prove: matching OS/kernel/Docker/driver state; a distinct, writable, non-network filesystem with enough space; expected RoCE interface/HCA state; one idle GB10; no compute process; and no recent kernel Xid. No address, host alias, filesystem source, device identifier, process ID, or command line is printed.

In [2]:
def run(command, **kwargs):
    return subprocess.run(command, check=True, text=True, **kwargs)

def parse_config(text):
    parsed = {}
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        parsed[key.strip()] = value.strip().strip("\"'")
    return parsed

if not RUN_LIVE:
    recorded = json.loads((RECIPE_DIR / "results" / "recorded-run.json").read_text(encoding="utf-8"))
    assert recorded["evidence_class"] == "MEASURED" and len(recorded["decode"]) == 4 and len(recorded["prefill"]) == 3
    print("RECORDED PREFLIGHT PASS — public structured evidence is complete; live node checks were not replayed.")
else:
    missing = [name for name, value in {"PIXELML_DGX_NODE_1": NODE_1, "PIXELML_DGX_NODE_2": NODE_2, "PIXELML_DGX_CONFIG_B64": CONFIG_B64, "PIXELML_API_TOKEN": API_TOKEN, "PIXELML_API_BASE": API_BASE}.items() if not value]
    assert not missing, "missing injected configuration: " + ", ".join(missing)
    assert NODE_1 != NODE_2, "node aliases must be distinct"
    parsed_url = urllib.parse.urlparse(API_BASE)
    assert parsed_url.scheme in {"http", "https"} and parsed_url.netloc and parsed_url.path.endswith("/v1"), "PIXELML_API_BASE must be an OpenAI-compatible /v1 base"
    config_text = base64.b64decode(CONFIG_B64, validate=True).decode("utf-8")
    CONFIG = parse_config(config_text)
    required = {"MODEL_DIR", "RANK0_ADDR", "DIST_PORT", "NCCL_SOCKET_IFNAME", "NCCL_IB_HCA", "NCCL_IB_GID_INDEX", "API_PORT", "CONTEXT_LENGTH", "MAX_RUNNING_REQUESTS", "MEM_FRACTION_STATIC"}
    absent = sorted(required - CONFIG.keys())
    assert not absent, "configuration payload is missing required recipe fields: " + ", ".join(absent)

    def probe(node, node_index):
        model_dir = shlex.quote(CONFIG["MODEL_DIR"])
        interface = shlex.quote(CONFIG["NCCL_SOCKET_IFNAME"])
        hca = shlex.quote(CONFIG["NCCL_IB_HCA"])
        rank0 = shlex.quote(CONFIG["RANK0_ADDR"])
        role_check = 'ip -o addr show dev "$interface" | grep -Fq " $rank0/"' if node_index == 1 else 'ip route get "$rank0" | grep -Fq "dev $interface"'
        script = f'''set -euo pipefail
model_dir={model_dir}
interface={interface}
hca={hca}
rank0={rank0}
for command in docker nvidia-smi findmnt ip sha256sum journalctl; do command -v "$command" >/dev/null; done
probe_path="$model_dir"
while [ ! -e "$probe_path" ]; do parent=$(dirname "$probe_path"); [ "$parent" != "$probe_path" ] || exit 20; probe_path="$parent"; done
[ -d "$probe_path" ] && [ -w "$probe_path" ]
free_bytes=$(df -PB1 "$probe_path" | awk 'NR==2 {{print $4}}')
[ "$free_bytes" -ge 171798691840 ]
fstype=$(findmnt -n -o FSTYPE -T "$probe_path")
case "$fstype" in nfs*|cifs|smb*|ceph*|glusterfs|fuse.sshfs) exit 21;; esac
fs_source=$(findmnt -n -o SOURCE -T "$probe_path")
fs_id=$(stat -f -c %i "$probe_path")
storage_sig=$(printf '%s' "$fstype|$fs_source|$fs_id" | sha256sum | awk '{{print $1}}')
[ "$(cat /sys/class/net/"$interface"/operstate)" = up ]
[ -d /sys/class/infiniband/"$hca" ]
grep -q ACTIVE /sys/class/infiniband/"$hca"/ports/1/state
{role_check}
docker info >/dev/null
docker_version=$(docker version --format '{{{{.Server.Version}}}}')
platform_sig=$(printf '%s' "$(sha256sum /etc/os-release | awk '{{print $1}}')|$(uname -r)|$docker_version" | sha256sum | awk '{{print $1}}')
gpu=$(nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu,driver_version --format=csv,noheader,nounits)
compute_rows=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader,nounits 2>/dev/null)
compute_count=$(printf '%s\n' "$compute_rows" | sed '/^$/d' | wc -l | tr -d ' ')
if kernel_window=$(journalctl -k --since '-1 hour' --no-pager 2>/dev/null); then journal_ok=1; else journal_ok=0; kernel_window=''; fi
xid_count=$(printf '%s\n' "$kernel_window" | grep -c 'NVRM: Xid' || true)
PLATFORM_SIG="$platform_sig" STORAGE_SIG="$storage_sig" FREE_BYTES="$free_bytes" FSTYPE="$fstype" GPU="$gpu" COMPUTE_COUNT="$compute_count" JOURNAL_OK="$journal_ok" XID_COUNT="$xid_count" python3 - <<'PY'
import json, os
name, total, used, util, temp, driver = [part.strip() for part in os.environ['GPU'].split(',')]
print(json.dumps({{'platform_sig': os.environ['PLATFORM_SIG'], 'storage_sig': os.environ['STORAGE_SIG'], 'free_bytes': int(os.environ['FREE_BYTES']), 'fs_type': os.environ['FSTYPE'], 'gpu_name': name, 'memory_total_mib': float(total), 'memory_used_mib': float(used), 'utilization_pct': float(util), 'temperature_c': float(temp), 'driver': driver, 'compute_count': int(os.environ['COMPUTE_COUNT']), 'journal_ok': os.environ['JOURNAL_OK'] == '1', 'xid_count': int(os.environ['XID_COUNT'])}}))
PY
'''
        return json.loads(run(["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", node, "bash", "-s"], input=script, capture_output=True).stdout)

    probes = [probe(NODE_1, 1), probe(NODE_2, 2)]
    assert probes[0]["platform_sig"] == probes[1]["platform_sig"], "nodes differ in OS/kernel/Docker runtime"
    assert probes[0]["gpu_name"] == probes[1]["gpu_name"] and probes[0]["driver"] == probes[1]["driver"], "nodes differ in GPU class or driver"
    assert probes[0]["storage_sig"] != probes[1]["storage_sig"], "model storage is not proven node-local and distinct"
    for index, probe_result in enumerate(probes, 1):
        assert probe_result["memory_total_mib"] >= 120000, f"node-{index}: expected GB10 memory class"
        assert probe_result["memory_used_mib"] < 2048 and probe_result["utilization_pct"] < 5 and probe_result["compute_count"] == 0, f"node-{index}: conflicting GPU workload"
        assert probe_result["temperature_c"] < 80, f"node-{index}: unsafe core temperature"
        assert probe_result["journal_ok"] and probe_result["xid_count"] == 0, f"node-{index}: kernel error gate unavailable or recent Xid present"
        print(f"node-{index}: GPU, thermals, node-local storage, runtime, interconnect, and conflict gates PASS")
    print("TWO-NODE PREFLIGHT PASS")

RECORDED PREFLIGHT PASS — public structured evidence is complete; live node checks were not replayed.


## Install the pinned recipe on both nodes

This cell creates isolated checkouts at the exact public commit. It proves each tree is clean before and after materializing the injected configuration and API token into ignored, recipe-declared private slots. Secret values travel only on subprocess standard input and are never included in arguments or outputs.

In [3]:
def checkout(source, destination):
    run(["git", "clone", "--no-checkout", source, str(destination)], capture_output=True)
    run(["git", "-C", str(destination), "checkout", "--detach", RECIPE_SHA], capture_output=True)
    assert run(["git", "-C", str(destination), "rev-parse", "HEAD"], capture_output=True).stdout.strip() == RECIPE_SHA
    assert run(["git", "-C", str(destination), "status", "--porcelain"], capture_output=True).stdout == ""

recorded_source = os.environ.get("PIXELML_RECORDED_RECIPE_SOURCE", RECIPE_URL)
checkout(recorded_source, CONTROLLER_REPO)

def declared_private_slot(repo, variable):
    contract = (repo / "scripts" / "common.sh").read_text(encoding="utf-8")
    line = next((line for line in contract.splitlines() if line.startswith(variable + "=")), "")
    match = re.search(r"REPO_ROOT\}[/]([^\"}]+)", line)
    assert match, f"pinned recipe does not declare {variable}"
    path = (repo / match.group(1)).resolve()
    assert repo.resolve() in path.parents, f"{variable} escapes isolated checkout"
    return path

CONTROLLER_SECRET_SLOT = None
MATERIALIZE = r'''import base64, json, os, re, subprocess, sys
from pathlib import Path
repo = Path(sys.argv[1])
payload = json.load(sys.stdin)
contract = (repo / "scripts" / "common.sh").read_text(encoding="utf-8")
def slot(variable):
    line = next((line for line in contract.splitlines() if line.startswith(variable + "=")), "")
    match = re.search(r"REPO_ROOT\}[/]([^\"}}]+)", line)
    if not match:
        raise RuntimeError(f"pinned recipe does not declare {variable}")
    path = repo / match.group(1)
    if path.is_absolute() or repo not in path.resolve().parents:
        raise RuntimeError(f"{variable} escapes isolated checkout")
    return path
config_slot = slot("ENV_FILE")
secret_slot = slot("SECRET_FILE")
config_slot.write_bytes(base64.b64decode(payload["config_b64"], validate=True))
secret_slot.write_text(payload["api_token"], encoding="utf-8")
os.chmod(config_slot, 0o600)
os.chmod(secret_slot, 0o600)
for path in (config_slot, secret_slot):
    subprocess.run(["git", "-C", str(repo), "check-ignore", "-q", str(path)], check=True)
'''

if RUN_LIVE:
    private_payload = json.dumps({"config_b64": CONFIG_B64, "api_token": API_TOKEN})
    run([sys.executable, "-c", MATERIALIZE, str(CONTROLLER_REPO)], input=private_payload, capture_output=True)
    CONTROLLER_SECRET_SLOT = declared_private_slot(CONTROLLER_REPO, "SECRET_FILE")
    assert run(["git", "-C", str(CONTROLLER_REPO), "check-ignore", "-q", str(CONTROLLER_SECRET_SLOT)], check=False).returncode == 0
    assert run(["git", "-C", str(CONTROLLER_REPO), "status", "--porcelain"], capture_output=True).stdout == ""
    for label, node in (("node-1", NODE_1), ("node-2", NODE_2)):
        workspace = run(["ssh", node, "mktemp", "-d"], capture_output=True).stdout.strip()
        assert workspace
        clone_script = f'''set -euo pipefail
git clone --no-checkout {shlex.quote(RECIPE_URL)} {shlex.quote(workspace)} >/dev/null 2>&1
git -C {shlex.quote(workspace)} checkout --detach {RECIPE_SHA} >/dev/null 2>&1
test "$(git -C {shlex.quote(workspace)} rev-parse HEAD)" = {RECIPE_SHA}
test -z "$(git -C {shlex.quote(workspace)} status --porcelain)"
'''
        run(["ssh", node, "bash", "-s"], input=clone_script, capture_output=True)
        run(["ssh", node, "python3", "-c", MATERIALIZE, workspace], input=private_payload, capture_output=True)
        clean = run(["ssh", node, "git", "-C", workspace, "status", "--porcelain"], capture_output=True).stdout
        assert clean == "", f"{label}: isolated checkout changed after private input injection"
        NODE_WORKSPACES[label] = workspace
    print("PINNED CLEAN CHECKOUT PASS — controller and both nodes")
else:
    print("PINNED CLEAN CHECKOUT PASS — recorded-mode controller verification")

PINNED CLEAN CHECKOUT PASS — recorded-mode controller verification


## Prepare model storage on each node

Live mode invokes the pinned resumable model-preparation script in parallel only after the distinct node-local storage gate passes. Recorded mode performs no model download.

In [4]:
if RUN_LIVE:
    processes = []
    for label, node in (("node-1", NODE_1), ("node-2", NODE_2)):
        workspace = NODE_WORKSPACES[label]
        command = f"cd {shlex.quote(workspace)} && ./scripts/prepare-model.sh"
        processes.append((label, subprocess.Popen(["ssh", node, command], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, text=True)))
    statuses = {label: process.wait() for label, process in processes}
    failures = {label: status for label, status in statuses.items() if status != 0}
    assert not failures, "node-local model preparation failed: " + ", ".join(sorted(failures))
    print("MODEL INTEGRITY PASS — independent node-local checkpoints")
else:
    print("MODEL PREPARATION NOT RUN — recorded mode is intentionally non-mutating.")

MODEL PREPARATION NOT RUN — recorded mode is intentionally non-mutating.


## Start and inspect the two-node service

Live mode starts rank 1, then rank 0, using the pinned lifecycle script and a bounded readiness loop. It submits no benchmark until the authenticated model-list response contains the expected public alias.

In [5]:
if RUN_LIVE:
    for label, node, rank in (("node-2", NODE_2, 1), ("node-1", NODE_1, 0)):
        workspace = NODE_WORKSPACES[label]
        command = f"cd {shlex.quote(workspace)} && ./scripts/start-node.sh {rank}"
        run(["ssh", node, command], capture_output=True)
    models_url = API_BASE + "/models"
    for attempt in range(120):
        try:
            request = urllib.request.Request(models_url, headers={"Authorization": f"Bearer {API_TOKEN}"})
            with urllib.request.urlopen(request, timeout=8) as response:
                payload = json.load(response)
            model_ids = [item.get("id") for item in payload.get("data", [])]
            if "qwen3.8-flash-next" in model_ids:
                break
        except Exception:
            pass
        time.sleep(15)
    else:
        raise RuntimeError("service did not become ready within 30 minutes")
    print("SERVICE READY — authenticated model alias inspected")
else:
    print("SERVICE START NOT RUN — recorded mode is intentionally non-mutating.")

SERVICE START NOT RUN — recorded mode is intentionally non-mutating.


## Benchmark and structured results

Live mode captures the pinned harness's JSONL, validates the complete C1/C4/C8/C16 and 1K/4K/16K ladders, and atomically replaces the structured run plus summary. Decode uses final `usage.completion_tokens`; every prefill sample retains the actual final `usage.prompt_tokens` used for its rate. Recorded mode rebuilds the same summary from the committed public snapshot.

In [6]:
recorded_run_path = RECIPE_DIR / "results" / "recorded-run.json"
summary_path = RECIPE_DIR / "results" / "summary.csv"
builder = REPO_ROOT / "scripts" / "build_recipe_results.py"
if RUN_LIVE:
    decode_jsonl = WORK_ROOT / "decode.jsonl"
    prefill_jsonl = WORK_ROOT / "prefill.jsonl"
    assert CONTROLLER_SECRET_SLOT and CONTROLLER_SECRET_SLOT.is_file()
    common = ["--base-url", API_BASE, "--secret-file", str(CONTROLLER_SECRET_SLOT)]
    decode_result = run([sys.executable, "scripts/smoke-benchmark.py", *common], cwd=CONTROLLER_REPO, capture_output=True)
    prefill_result = run([sys.executable, "scripts/prefill-benchmark.py", *common], cwd=CONTROLLER_REPO, capture_output=True)
    decode_jsonl.write_text(decode_result.stdout, encoding="utf-8")
    prefill_jsonl.write_text(prefill_result.stdout, encoding="utf-8")
    result = run([sys.executable, str(builder), "--decode-jsonl", str(decode_jsonl), "--prefill-jsonl", str(prefill_jsonl), "--evidence", PINS["evidence"], "--raw-output", str(recorded_run_path), "--summary", str(summary_path)], capture_output=True)
else:
    result = run([sys.executable, str(builder), "--recorded-run", str(recorded_run_path), "--summary", str(summary_path)], capture_output=True)
print(result.stdout.strip())
rows = list(csv.DictReader(summary_path.open(encoding="utf-8")))
print("metric | x | requests | samples | prompt_tokens | completion_tokens | duration_s | tok_s")
print("--- | ---: | ---: | ---: | ---: | ---: | ---: | ---:")
for row in rows:
    print(" | ".join(row[column] for column in ("metric", "x", "request_count", "sample_count", "prompt_tokens", "completion_tokens", "duration_seconds", "tok_s")))

validated 4 decode and 3 prefill levels
metric | x | requests | samples | prompt_tokens | completion_tokens | duration_s | tok_s
--- | ---: | ---: | ---: | ---: | ---: | ---: | ---:
decode | 1 | 1 | 1 |  | 192 | 4.038 | 47.54
decode | 4 | 4 | 1 |  | 750 | 8.567 | 87.55
decode | 8 | 8 | 1 |  | 1536 | 9.711 | 158.17
decode | 16 | 16 | 1 |  | 3072 | 11.156 | 275.37
prefill | 1024 | 1 | 3 | 1053 | 1 | 0.4524 | 2327.78
prefill | 4096 | 1 | 3 | 4117 | 1 | 1.4924 | 2758.65
prefill | 16384 | 1 | 3 | 16407 | 1 | 5.5427 | 2960.12


In [7]:
run([sys.executable, str(REPO_ROOT / "scripts" / "render_recipe_chart.py"), "--spec", str(RECIPE_DIR / "chart-spec.json"), "--data", str(summary_path), "--output", str(RECIPE_DIR / "assets" / "performance.png")])
print("CHART PASS — rendered only from validated results/summary.csv")

CHART PASS — rendered only from validated results/summary.csv


## Try your own prompt

Edit `PROMPT` below. In live mode the literal `curl` cell fails on a non-2xx response, an empty answer, a missing/non-numeric usage field, or zero completion tokens. The token is passed to curl over standard input rather than a process argument.

In [8]:
PROMPT = "Write a compact Python function that validates a topological ordering. Return code only."
os.environ["PIXELML_PROMPT"] = PROMPT
os.environ["PIXELML_API_BASE"] = API_BASE
os.environ["PIXELML_API_TOKEN"] = API_TOKEN
c1 = json.loads(recorded_run_path.read_text(encoding="utf-8"))["decode"][0]
os.environ["PIXELML_RECORDED_RESPONSE"] = json.dumps({"mode": "recorded", "response": "Final prompt text was not retained in the public measured receipt; live mode prints a fresh inspected answer.", "usage": {"completion_tokens": c1["completion_tokens"], "token_basis": "final API usage from the measured C1 receipt"}})
print(PROMPT)

Write a compact Python function that validates a topological ordering. Return code only.


In [9]:
%%bash
set -euo pipefail
if [ "${PIXELML_RUN_LIVE:-0}" != "1" ]; then
  printf '%s\n' "$PIXELML_RECORDED_RESPONSE" | jq -e 'select(.response and (.usage | type == "object") and (.usage.completion_tokens > 0))'
  exit 0
fi
BODY_FILE=$(mktemp)
RESPONSE_FILE=$(mktemp)
trap 'rm -f "$BODY_FILE" "$RESPONSE_FILE"' EXIT
jq -n --arg prompt "$PIXELML_PROMPT" '{model:"qwen3.8-flash-next",messages:[{role:"user",content:$prompt}],max_tokens:256,temperature:0.0,reasoning_effort:"low",stream:false}' >"$BODY_FILE"
HTTP_CODE=$(curl --silent --show-error --fail-with-body --output "$RESPONSE_FILE" --write-out '%{http_code}' --config - <<EOF
url = "$PIXELML_API_BASE/chat/completions"
header = "Authorization: Bearer $PIXELML_API_TOKEN"
header = "Content-Type: application/json"
data = "@$BODY_FILE"
EOF
)
case "$HTTP_CODE" in 2??) ;; *) echo "unexpected HTTP status" >&2; exit 1;; esac
jq -e '(.choices | type == "array" and length > 0) and (.choices[0].message.content | type == "string" and length > 0) and (.usage | type == "object") and (.usage.prompt_tokens | type == "number" and . >= 0) and (.usage.completion_tokens | type == "number" and . > 0) and (.usage.total_tokens | type == "number" and . > 0)' "$RESPONSE_FILE" >/dev/null
jq -r '.choices[0].message.content' "$RESPONSE_FILE"
printf '\nusage:\n'
jq -e '.usage | {prompt_tokens, completion_tokens, total_tokens}' "$RESPONSE_FILE"

{
  "mode": "recorded",
  "response": "Final prompt text was not retained in the public measured rec

eipt; live mode prints a fresh inspected answer.",
  "usage": {
    "completion_tokens": 192,
    "t

oken_basis": "final API usage from the measured C1 receipt"
  }
}
